# Order Book Simulator Demo

This notebook demonstrates how to use the OrderBookSim library to create and run a limit order book simulation.

In [1]:
# Add project root to Python path for imports
import sys
import os
sys.path.append(os.path.abspath('..'))

# Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from IPython.display import display

# Set plot styles
plt.style.use('ggplot')
sns.set_theme()
%matplotlib inline

## Import Order Book Simulator Components

In [2]:
# Import core simulator components
from sim.simulator import Simulator
from sim.config import get_demo_config
from core.order_book import OrderSide, OrderType, Order

# Import agent types
from agents.market_maker import MarketMaker
from agents.aggressive_trader import AggressiveTrader
from agents.noise_trader import NoiseTrader

# Import visualization functions
from app.visuals import create_depth_chart, create_price_chart, create_trades_chart

## Create and Configure the Simulation

We'll use the demo configuration, which includes a mix of different agent types and reasonable market parameters.

In [3]:
# Get demo configuration
config = get_demo_config()

# Print configuration details
print(f"Simulation Configuration:\n")
print(f"Market Settings:")
print(f"  Tick Size: {config.tick_size}")
print(f"  Initial Price: {config.initial_price}\n")

print(f"Agent Configuration:")
print(f"  Market Makers: {config.market_makers.count}")
print(f"  Aggressive Traders: {config.aggressive_traders.count}")
print(f"  Noise Traders: {config.noise_traders.count}\n")

print(f"Simulation Settings:")
print(f"  Max Ticks: {config.max_ticks}")
print(f"  Step Delay: {config.step_delay}s")

Simulation Configuration:

Market Settings:
  Tick Size: 0.01
  Initial Price: 100.0

Agent Configuration:
  Market Makers: 3
  Aggressive Traders: 4
  Noise Traders: 8

Simulation Settings:
  Max Ticks: 500
  Step Delay: 0.1s


## Create a Simulator with Agents

In [4]:
def create_simulator(config):
    """Create a simulator with agents based on configuration."""
    # Create simulator with config parameters
    simulator = Simulator(
        tick_size=config.tick_size,
        initial_price=config.initial_price,
        log_trades=True
    )
    
    # Dictionary for agent names
    agent_names = {}
    
    # Add market makers
    for i in range(config.market_makers.count):
        mm = MarketMaker(
            name=f"MM-{i+1}",
            quote_spread=config.market_makers.quote_spread,
            quote_volume=config.market_makers.quote_volume,
            max_position=config.market_makers.max_position,
            inventory_skew_factor=config.market_makers.inventory_skew_factor,
            tick_size=config.tick_size,
            skew_quotes=config.market_makers.skew_quotes
        )
        simulator.add_agent(mm)
        agent_names[mm.agent_id] = mm.name
    
    # Add aggressive traders
    for i in range(config.aggressive_traders.count):
        at = AggressiveTrader(
            name=f"Aggr-{i+1}",
            order_rate=config.aggressive_traders.order_rate,
            trade_size_range=(
                config.aggressive_traders.trade_size_min,
                config.aggressive_traders.trade_size_max
            ),
            max_position=config.aggressive_traders.max_position,
            position_influence=config.aggressive_traders.position_influence
        )
        simulator.add_agent(at)
        agent_names[at.agent_id] = at.name
    
    # Add noise traders
    for i in range(config.noise_traders.count):
        nt = NoiseTrader(
            name=f"Noise-{i+1}",
            order_rate=config.noise_traders.order_rate,
            limit_order_prob=config.noise_traders.limit_order_prob,
            price_range_factor=config.noise_traders.price_range_factor,
            cancel_rate=config.noise_traders.cancel_rate,
            size_range=(
                config.noise_traders.size_min,
                config.noise_traders.size_max
            ),
            tick_size=config.tick_size
        )
        simulator.add_agent(nt)
        agent_names[nt.agent_id] = nt.name
    
    return simulator, agent_names

# Create the simulator with our config
simulator, agent_names = create_simulator(config)
print(f"Created simulator with {len(simulator.agents)} agents.")

Created simulator with 15 agents.


## Run the Simulation

Now let's run the simulation for 100 ticks.

In [5]:
# Run the simulation for 100 ticks
num_ticks = 100
print(f"Running simulation for {num_ticks} ticks...")

results = simulator.run(num_steps=num_ticks, step_delay=0.0)

print(f"Simulation complete!")
print(f"Trades executed: {results['trade_count']}")
print(f"Total volume: {results['total_volume']}")
print(f"Final mid price: {results['final_mid_price']:.2f}")

Running simulation for 100 ticks...
Simulation complete!
Trades executed: 459
Total volume: 4211
Final mid price: 98.89


## Visualize the Results

In [6]:
# Get order book state
bids_df, asks_df = simulator.order_book.get_order_book_as_dataframe()

# Create depth chart
fig = create_depth_chart(bids_df, asks_df)
fig.show()

In [7]:
# Get price history
history_df = simulator.get_history_as_dataframe()

# Create price chart
fig = create_price_chart(history_df)
fig.show()

In [8]:
# Get trade log
trades_df = simulator.get_trade_log_as_dataframe()

# Show trade activity chart
if not trades_df.empty:
    fig = create_trades_chart(trades_df)
    fig.show()
else:
    print("No trades executed yet.")

## Agent Performance Analysis

In [9]:
# Create P&L comparison chart
fig = go.Figure()

# Add P&L line for each agent
for agent_id, agent in simulator.agents.items():
    metrics_df = simulator.get_agent_metrics_as_dataframe(agent_id)
    if metrics_df is not None and 'pnl' in metrics_df.columns:
        name = agent_names.get(agent_id, f"Agent {agent_id[:8]}")
        fig.add_trace(go.Scatter(
            x=metrics_df['tick'],
            y=metrics_df['pnl'],
            mode='lines',
            name=name
        ))

# Update layout
fig.update_layout(
    title="Agent P&L Comparison",
    xaxis_title="Tick Number",
    yaxis_title="P&L",
    hovermode="x unified"
)

fig.show()

In [10]:
# Check final positions and P&L for each agent
agent_results = []

for agent_id, agent in simulator.agents.items():
    agent_type = type(agent).__name__
    name = agent_names.get(agent_id, f"Agent {agent_id[:8]}")
    
    # Calculate P&L
    mark_price = simulator.order_book.mid_price or simulator.initial_price
    pnl = agent.update_pnl(mark_price)
    
    agent_results.append({
        "Name": name,
        "Type": agent_type,
        "Position": agent.position,
        "P&L": pnl,
        "Trades": agent.trades_executed,
        "Volume": agent.volume_traded
    })

# Create DataFrame and display
results_df = pd.DataFrame(agent_results)
display(results_df)

,Name,Type,Position,P&L,Trades,Volume
0,MM-1,MarketMaker,-6,-17.65,256,2466
1,MM-2,MarketMaker,25,-12.54,254,2555
2,MM-3,MarketMaker,19,-15.60,270,2519
3,Aggr-1,AggressiveTrader,-24,18.32,23,166
4,Aggr-2,AggressiveTrader,-14,30.95,29,218
5,Aggr-3,AggressiveTrader,26,-22.41,24,160
6,Aggr-4,AggressiveTrader,21,-4.28,19,131
7,Noise-1,NoiseTrader,4,-0.92,3,16
8,Noise-2,NoiseTrader,-1,3.59,6,23
9,Noise-3,NoiseTrader,14,-5.62,5,24


## Export Results

You can export the simulation results to CSV files for further analysis.

In [11]:
# Create data directory if it doesn't exist
os.makedirs("../data", exist_ok=True)

# Export data
export_path = "../data/notebook_demo"
export_results = simulator.export_results(export_path)

# Show what files were created
for name, success in export_results.items():
    if success:
        print(f"✅ Exported {name} data")
    else:
        print(f"❌ Failed to export {name} data")

✅ Exported history data
✅ Exported trades data
✅ Exported agent_mm-1 data
✅ Exported agent_mm-2 data
✅ Exported agent_mm-3 data
✅ Exported agent_aggr-1 data
✅ Exported agent_aggr-2 data
✅ Exported agent_aggr-3 data
✅ Exported agent_aggr-4 data
✅ Exported agent_noise-1 data
✅ Exported agent_noise-2 data
✅ Exported agent_noise-3 data
✅ Exported agent_noise-4 data
✅ Exported agent_noise-5 data
✅ Exported agent_noise-6 data
✅ Exported agent_noise-7 data
✅ Exported agent_noise-8 data
✅ Exported order_book_bids data
✅ Exported order_book_asks data
